In [59]:
from pathlib import Path
import json
import pandas as pd

BASE_DIR = Path(".").resolve()
DA_MAPPINGS_DIR = BASE_DIR / "da_mappings" / "da_mappings_obx"


In [60]:
from IPython.display import display, HTML

rows = []

for fp in sorted(DA_MAPPINGS_DIR.glob("*.json")):
    data = json.loads(fp.read_text(encoding="utf-8"))
    firm_id = fp.stem

    for var_block in data.get("variables", []):
        variable = var_block.get("variable")
        final_choice = var_block.get("final_choice", [])

        if final_choice:  # only keep non-empty final_choice
            for ch in final_choice:
                rows.append({
                    "firm_id": firm_id,
                    "variable": variable,
                    "sheet_name": ch.get("sheet_name", ""),
                    "row_label": ch.get("row_label", ""),
                    "needs_manual_review": var_block.get("needs_manual_review", False),
                    "notes": var_block.get("notes", ""),
                })

df_nonempty = pd.DataFrame(rows)

if df_nonempty.empty:
    print("No non-empty final_choice found.")
else:
    df_show = df_nonempty.sort_values([ "firm_id","variable"]).reset_index(drop=True)

    html_table = df_show.to_html(index=False, escape=False)

    display(HTML(f"""
    <div style="
        max-height: 600px;
        overflow-y: auto;
        overflow-x: auto;
        border: 1px solid #ccc;
        padding: 6px;
        background: black;
    ">
        {html_table}
    </div>
    """))

firm_id,variable,sheet_name,row_label,needs_manual_review,notes
20202.OL,COGS_DA,Income Statement,Depreciation,False,OK. Selected the standalone operating depreciation line for COGS_DA because it sits outside the already selected COGS rows and no more specific COGS depreciation component is shown.
ABL.OL,XSGA_DA,Income Statement,Depreciation & Amortization,False,"Manually Added. A separate operating line 'Depreciation, Amortization & Impairment' suggests D&A is not embedded in the selected XSGA rows, but that row also includes impairment, which must be ignored. The pure D&A component rows shown are only partial/limited-year breakouts, so selecting them risks undercounting while selecting the summary risks including impairment. Manual review is needed."
AFGA.OL,XSGA_DA,Income Statement,Depreciation of PPE,False,"Manually Changed. The selected XSGA bucket is Selling, General & Admin plus Operating Expenses. Visible depreciation rows are presented separately below those operating expense rows, so they appear not already embedded in the selected XSGA final_choice and should be added separately. No dedicated R&D row is selected in XRD, so there is no R&D-specific D&A interaction to apply here. Mixed impairment/amortisation rows were otherwise avoided where possible to reduce non-D&A contamination."
AFGA.OL,XSGA_DA,Income Statement,Depreciation and impairment of right of use assets,False,"Manually Changed. The selected XSGA bucket is Selling, General & Admin plus Operating Expenses. Visible depreciation rows are presented separately below those operating expense rows, so they appear not already embedded in the selected XSGA final_choice and should be added separately. No dedicated R&D row is selected in XRD, so there is no R&D-specific D&A interaction to apply here. Mixed impairment/amortisation rows were otherwise avoided where possible to reduce non-D&A contamination."
AFK.OL,XSGA_DA,Income Statement,Depreciation,False,"Manually Added. The selected XSGA bucket consists of 'Employee benefit expenses', 'Stock-Based Compensation in Selling, General & Administrative Expenses', and 'Other Operating Expenses'. Generic 'Depreciation'/'Amortisation' rows are visible but not clearly attributable to XSGA versus other operating buckets, so they are excluded conservatively. However, 'Amortization excluding Goodwill in Research & Development Expenses' is an explicit operating R&D D&A row, no separate above-EBIT R&D expense row is present, and XRD is empty, so this row should be added separately to XSGA_DA per the special R&D interaction rule."
AFK.OL,XSGA_DA,Income Statement,Amortisation,False,"Manually Added. The selected XSGA bucket consists of 'Employee benefit expenses', 'Stock-Based Compensation in Selling, General & Administrative Expenses', and 'Other Operating Expenses'. Generic 'Depreciation'/'Amortisation' rows are visible but not clearly attributable to XSGA versus other operating buckets, so they are excluded conservatively. However, 'Amortization excluding Goodwill in Research & Development Expenses' is an explicit operating R&D D&A row, no separate above-EBIT R&D expense row is present, and XRD is empty, so this row should be added separately to XSGA_DA per the special R&D interaction rule."
AKAST.OL,XSGA_DA,Income Statement,Total Depreciation & Amortization (Combined),False,"The selected XSGA_COMPONENTS bucket consists of specific operating cost components that do not explicitly include D&A, so a separate SG&A D&A add-back is possible in principle. However, the excerpt provides only generic depreciation/amortisation lines with no functional attribution to SG&A, COGS, or R&D. Since R&D is already visible as a separate row and selected in XRD, there is no basis to add any R&D-related amortisation separately. Conservative output is no selection pending manual review to avoid misallocation or double counting."
AKBM.OL,XSGA_DA,Income Statement,"Amortization of Licenses, Copyrights, Property Rights & Designs",False,"Manually Added. Se

In [61]:
df_manual = df_nonempty[df_nonempty["needs_manual_review"] == True]

print(f"Number of rows:                    {len(df_nonempty)}")
print(f"Unique tickers (firm_id):          {df_nonempty['firm_id'].nunique()}")
print(f"\nNeeds manual review (total rows):  {len(df_manual)}")
print(f"Needs manual review (unique tickers): {df_manual['firm_id'].nunique()}")
print(f"\nAll tickers: {sorted(df_nonempty['firm_id'].unique().tolist())}")

Number of rows:                    159
Unique tickers (firm_id):          132

Needs manual review (total rows):  0
Needs manual review (unique tickers): 0

All tickers: ['20202.OL', 'ABL.OL', 'AFGA.OL', 'AFK.OL', 'AKAST.OL', 'AKBM.OL', 'AKH.OL', 'AKRBP.OL', 'AKSOA.OL', 'AKVA.OL', 'ARCHA.OL', 'ARRA.OL', 'ASA.OL', 'ATEA.OL', 'AUSS.OL', 'AUTO.OL', 'AZT.OL', 'BAKKA.OL', 'BNOR.OL', 'BONHR.OL', 'BOR.OL', 'BOUV.OL', 'BRGB.OL', 'BWE.OL', 'BWLPG.OL', 'CAPSL.OL', 'CAVEN.OL', 'CLOUD.OL', 'CMBT.OL', 'CONTX.OL', 'CRNA.OL', 'DOFG.OL', 'EIOF.OL', 'ELABS.OL', 'ELK.OL', 'ELMRA.OL', 'ELO.OL', 'EMGS.OL', 'ENDUR.OL', 'ENH.OL', 'ENSU.OL', 'EPR.OL', 'EQNR.OL', 'EQVA.OL', 'FRO.OL', 'GENT.OL', 'GOD.OL', 'GSFG.OL', 'GYL.OL', 'HAFNI.OL', 'HAUTO.OL', 'HAVI.OL', 'HBC.OL', 'HEX.OL', 'HPUR.OL', 'HSHP.OL', 'HYPRO.OL', 'IDEX.OL', 'ITERA.OL', 'IWS.OL', 'JINJ.OL', 'KCCK.OL', 'KID.OL', 'KIT.OL', 'KOA.OL', 'KOG.OL', 'KOMPLK.OL', 'LINK.OL', 'LSG.OL', 'MEDI.OL', 'MGN.OL', 'MOWI.OL', 'MPCC.OL', 'MULTI.OL', 'NAPA.OL', 'NAS.

In [62]:
rows = []

for fp in sorted(DA_MAPPINGS_DIR.glob("*.json")):
    firm_id = fp.stem
    data = json.loads(fp.read_text(encoding="utf-8"))

    for v in data.get("variables", []):
        if v.get("needs_manual_review", False):
            final_choice = v.get("final_choice", [])
            # make final_choice readable
            final_str = "; ".join([f"{x['sheet_name']}::{x['row_label']}" for x in final_choice]) if final_choice else ""
            rows.append({
                "firm_id": firm_id,
                "variable": v.get("variable", ""),
                "notes": v.get("notes", ""),
                "final_choice": final_str,
                "num_candidates": len(v.get("candidates", [])),
            })

if len(rows) == 0:
    print("No manual reviews needed. All mappings have been reviewed and finalized.")
else:
    df_review = pd.DataFrame(rows).sort_values(["firm_id", "variable"]).reset_index(drop=True)

    print(f"Manual review needed for {len(df_review)} firm-variable mappings across {df_review['firm_id'].nunique() if len(df_review) else 0} firms.")
    display(HTML(f"""
    <div style="max-height: 450px; overflow-y: auto; border: 1px solid #ddd; padding: 6px;">
    {df_review.to_html(index=False)}
    </div>
    """))

Manual review needed for 6 firm-variable mappings across 3 firms.


firm_id,variable,notes,final_choice,num_candidates
SALM.OL,COGS_DA,"An explicit operating-level D&A summary exists separately from Cost of goods sold, so some D&A may need separate treatment. But the excerpt does not identify which portion belongs to COGS versus SG&A, and using the total D&A row for COGS would likely overstate COGS_DA.",,4
SALM.OL,XSGA_DA,"A separate D&A summary row is disclosed at the operating level, but it is not attributable specifically to Salary and personnel expenses, Other Operating Expenses, or Research & Development. Because the earlier mapping already includes a separate R&D row in both XSGA_COMPONENTS and XRD, adding any non-attributed D&A row to XSGA_DA would be unsafe without a bucket split.",,4
VEND.OL,COGS_DA,"The excerpt shows only a single separate operating D&A summary plus components, with no indication of how much belongs to cost of goods/services sold versus overhead. Relative to the selected COGS rows ('Costs of goods and services sold' / 'Raw materials and finished goods'), there is no explicit COGS-tagged D&A row to add separately. Conservative output is empty to avoid misallocating total operating D&A into COGS.",,6
VEND.OL,XSGA_DA,"The operating section presents 'Depreciation and amortisation' as a separate line outside the selected XSGA component rows, so D&A does appear not embedded in those selected rows. But the statement does not identify what portion belongs to SG&A versus COGS, and there is no SG&A-specific D&A row. To avoid double counting or misclassification, no row is selected automatically; manual review is needed if model construction requires assigning total operating D&A to a bucket.",,6
YAR.OL,COGS_DA,"The excerpt shows a separate operating-level 'Depreciation and amortization' line, so D&A does not appear embedded in the selected COGS row. But there is no row indicating which portion relates to COGS versus SG&A. To avoid speculative allocation and double counting, no COGS_DA row is selected.",,4
YAR.OL,XSGA_DA,"A separate operating 'Depreciation and amortization' line exists outside the selected SG&A row, so D&A may sit alongside SG&A rather than inside it. However, the statement does not identify any SG&A-specific D&A (nor R&D-specific amortization), and the visible R&D row was mapped from a supplemental section below operating profit. Conservative output is no selection pending manual review to prevent misallocation or double counting.",,4
